In [2]:
import pandas as pd
import numpy as np
import math
from pathlib import Path
import matplotlib.pyplot as plt
from itertools import combinations
import pyarrow.parquet as pq
import re

In [3]:


# --- directories for each year ---
dir_2024 = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\silver\rides\ride_year=2024")
dir_2025 = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\silver\rides\ride_year=2025")

for p in (dir_2024, dir_2025):
    if not p.exists():
        raise FileNotFoundError(f"Missing rides path: {p}")

# --- helper: list parquet files under a directory (recursively) ---
def list_parquet_files(base: Path):
    # include .parquet files and files under partitioned folders
    files = list(base.rglob("*.parquet"))
    # if directory itself is a parquet dataset (pyarrow style), this will be empty;
    # in that case return the directory path so pd.read_parquet can read the dataset
    if not files:
        return [base]
    return files

files_2024 = list_parquet_files(dir_2024)
files_2025 = list_parquet_files(dir_2025)
all_inputs = files_2024 + files_2025

print(f"Found {len(files_2024)} parquet inputs in 2024 dir and {len(files_2025)} in 2025 dir (total {len(all_inputs)})")

# --- read files in a memory-conscious loop and collect DataFrames ---
dfs = []
seen_schemas = set()
for src in all_inputs:
    try:
        # If src is a directory (dataset), pandas can read it directly
        if Path(src).is_dir():
            df = pd.read_parquet(src, engine="pyarrow")
        else:
            # file path
            df = pd.read_parquet(src, engine="pyarrow")
    except Exception as e:
        # try pyarrow dataset read as fallback
        try:
            table = pq.ParquetDataset(str(src)).read()
            df = table.to_pandas()
        except Exception as e2:
            raise RuntimeError(f"Failed to read {src}: {e}; fallback error: {e2}")
    # normalize column names to str and strip whitespace
    df.columns = [str(c).strip() for c in df.columns]
    dfs.append(df)
    seen_schemas.add(tuple(sorted(df.columns)))

# --- align schemas: union of all columns, fill missing with NaN ---
if not dfs:
    raise ValueError("No parquet files found to read.")

all_cols = sorted({c for df in dfs for c in df.columns})
aligned = []
for df in dfs:
    missing = [c for c in all_cols if c not in df.columns]
    if missing:
        # add missing columns with NaN
        for c in missing:
            df[c] = pd.NA
    # reorder columns consistently
    df = df[all_cols]
    aligned.append(df)

# --- concat into single rides_df ---
rides_df = pd.concat(aligned, axis=0, ignore_index=True, copy=False)

# --- quick sanity checks and summary ---
print("Concatenated rides_df rows:", f"{len(rides_df):,}")
print("Columns count:", len(rides_df.columns))
print("Sample schema preview:")
print(rides_df.dtypes.head(20))

# optional: show how many rows came from each source if you want provenance
# (only if original DataFrames had a consistent index)
# Example: add source column during read if provenance is needed

Found 27 parquet inputs in 2024 dir and 29 in 2025 dir (total 56)
Concatenated rides_df rows: 27,523,825
Columns count: 18
Sample schema preview:
end_canonical_station_id              object
end_coord_key                         object
end_station_district                  object
end_station_key                       object
end_station_latitude                 float64
end_station_longitude                float64
end_station_name                      object
end_station_name_norm                 object
end_time_ms                   datetime64[ns]
start_canonical_station_id            object
start_coord_key                       object
start_station_district                object
start_station_key                     object
start_station_latitude               float64
start_station_longitude              float64
start_station_name                    object
start_station_name_norm               object
start_time_ms                 datetime64[ns]
dtype: object


In [4]:
# --- 2. Identify candidate fields ---
candidate_station_cols = [c for c in rides_df.columns if "canonical_station_id" in c]
candidate_time_cols = [c for c in rides_df.columns if any(k in c.lower() for k in ["time", "date", "ts", "timestamp"])]

print("\nCandidate canonical station columns:", candidate_station_cols)
print("Candidate time/date columns:", candidate_time_cols)

# --- 3. Select pilot station ---
if "start_canonical_station_id" not in rides_df.columns:
    raise ValueError("No start_canonical_station_id found in schema.")

pilot_station_id = (
    rides_df["start_canonical_station_id"]
    .dropna()
    .sort_values()
    .iloc[0]
)

print("\nPilot canonical_station_id selected:", pilot_station_id)

# --- 4. Quick coverage check ---
coverage_counts = {
    "start_id_non_null": rides_df["start_canonical_station_id"].notna().sum(),
    "end_id_non_null": rides_df["end_canonical_station_id"].notna().sum(),
    "total_rows": len(rides_df),
}
print("\nCoverage counts:", coverage_counts)

# --- 5. Sample preview ---
print("\nSample rows:")
print(rides_df.head(10))


Candidate canonical station columns: ['end_canonical_station_id', 'start_canonical_station_id']
Candidate time/date columns: ['end_time_ms', 'start_time_ms']

Pilot canonical_station_id selected: STN_0001

Coverage counts: {'start_id_non_null': np.int64(27517222), 'end_id_non_null': np.int64(27394282), 'total_rows': 27523825}

Sample rows:
  end_canonical_station_id         end_coord_key  \
0                 STN_0001  45.524236,-73.581552   
1                 STN_0132  45.520516,-73.567696   
2                 STN_0076  45.529707,-73.569981   
3                 STN_0134  45.512168,-73.554816   
4                 STN_0010  45.523845,-73.552366   
5                 STN_0340  45.521188,-73.553789   
6                 STN_0111  45.524378,-73.560549   
7                 STN_0541  45.539900,-73.599362   
8                 STN_0149  45.480208,-73.577599   
9                 STN_0621  45.542232,-73.554444   

              end_station_district  \
0            Le Plateau-Mont-Royal   
1       

In [5]:


# Load GBFS feeds with pandas
info_url   = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"
status_url = "https://gbfs.velobixi.com/gbfs/2-2/en/station_status.json"

info_data   = pd.read_json(info_url)
status_data = pd.read_json(status_url)

# Normalize into DataFrames
info_df   = pd.json_normalize(info_data['data']['stations'])
status_df = pd.json_normalize(status_data['data']['stations'])

# Output column headers
print("info_df columns:")
print(info_df.columns.tolist())

print("\nstatus_df columns:")
print(status_df.columns.tolist())


print(rides_df.columns.tolist())



# Merge live info + status
live_stations_df = pd.merge(info_df, status_df, on="station_id", how="inner")

# --- Load canonical mapping (your CSV) ---
mapping_file = r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\silver\station_cleaning\station_direct_match_mapping_csv\part-00000-05727353-4315-467e-811a-270e04e821e6-c000.csv"
canonical_mapping_df = pd.read_csv(mapping_file)

info_df columns:
['station_id', 'external_id', 'name', 'short_name', 'lat', 'lon', 'rental_methods', 'capacity', 'electric_bike_surcharge_waiver', 'is_charging', 'eightd_has_key_dispenser', 'has_kiosk']

status_df columns:
['station_id', 'num_bikes_available', 'num_ebikes_available', 'vehicle_types_available', 'num_bikes_disabled', 'num_docks_available', 'num_docks_disabled', 'is_installed', 'is_renting', 'is_returning', 'last_reported', 'eightd_has_available_keys', 'is_charging']
['end_canonical_station_id', 'end_coord_key', 'end_station_district', 'end_station_key', 'end_station_latitude', 'end_station_longitude', 'end_station_name', 'end_station_name_norm', 'end_time_ms', 'start_canonical_station_id', 'start_coord_key', 'start_station_district', 'start_station_key', 'start_station_latitude', 'start_station_longitude', 'start_station_name', 'start_station_name_norm', 'start_time_ms']


In [6]:
# --- Compute median capacity ---
if "capacity" in info_df.columns:
    capacity_override = int(info_df["capacity"].median())
    print("Capacity override set to median capacity:", capacity_override)
else:
    raise ValueError("No 'capacity' column found in info_df")

# --- Initial bikes override (optional) ---
initial_bikes_override = None  # Example: set to 20 if known snapshot exists

# --- Resolve station capacity ---
station_capacity = float(capacity_override)

# --- Resolve initial bikes ---

initial_bikes = (
    int(initial_bikes_override)
    if initial_bikes_override is not None
    else math.floor(0.5 * station_capacity)
)


print("Station capacity used:", station_capacity)
print("Initial bikes used:", initial_bikes)
print("Capacity source: median from GBFS info_df")

Capacity override set to median capacity: 22
Station capacity used: 22.0
Initial bikes used: 11
Capacity source: median from GBFS info_df


In [7]:

# --- Define mapping directory and file ---
mapping_csv_dir = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\silver\station_cleaning\station_direct_match_mapping_csv")
mapping_file = mapping_csv_dir / "part-00000-05727353-4315-467e-811a-270e04e821e6-c000.csv"

# --- Load mapping CSV ---
if not mapping_file.exists():
    raise FileNotFoundError(f"Expected mapping file not found: {mapping_file}")

canonical_mapping_df = pd.read_csv(mapping_file)
mapping_source = str(mapping_file)

# --- Validate required columns ---
required_map_cols = [
    "canonical_station_id",
    "canonical_lat",
    "canonical_lon",
    "normalized_name",
    "coord_key",
]
missing_map_cols = [c for c in required_map_cols if c not in canonical_mapping_df.columns]
if missing_map_cols:
    raise ValueError(f"Missing required mapping columns: {missing_map_cols}")

# --- Report summary ---
print("Mapping source:", mapping_source)
print("Mapping rows:", f"{len(canonical_mapping_df):,}")
print("Canonical stations:", canonical_mapping_df["canonical_station_id"].nunique())

# --- Preview required columns ---
print("\nPreview of required mapping columns:")
print(canonical_mapping_df[required_map_cols].tail(10))

Mapping source: C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\silver\station_cleaning\station_direct_match_mapping_csv\part-00000-05727353-4315-467e-811a-270e04e821e6-c000.csv
Mapping rows: 1,901
Canonical stations: 1422

Preview of required mapping columns:
     canonical_station_id  canonical_lat  canonical_lon  \
1891             STN_1079      45.639123     -73.596112   
1892             STN_1174      45.640003     -73.490113   
1893             STN_1148      45.640138     -73.492357   
1894             STN_1092      45.641415     -73.499957   
1895             STN_1311      45.641427     -73.500654   
1896             STN_1088      45.649546     -73.491915   
1897             STN_1076      45.651406     -73.500413   
1898             STN_1151      45.692233     -73.637031   
1899             STN_1216      45.697815     -73.654065   
1900             STN_1220      45.702349     -73.639578   

                                        normalized_name             coo

In [8]:
# --- CanonicalStationResolver with resolve_by_coord helper ---
class CanonicalStationResolver:
    def __init__(self, canonical_mapping: pd.DataFrame, max_name_coord_km: float = 0.05, max_nearest_km: float = 0.05):
        required_cols = {
            "canonical_station_id", "canonical_lat", "canonical_lon", "normalized_name", "coord_key"
        }
        missing = required_cols - set(canonical_mapping.columns)
        if missing:
            raise ValueError(f"canonical_mapping is missing columns: {sorted(missing)}")

        self.max_name_coord_km = float(max_name_coord_km)
        self.max_nearest_km = float(max_nearest_km)

        self.canon_unique_df = (
            canonical_mapping[["canonical_station_id", "canonical_lat", "canonical_lon", "normalized_name", "coord_key"]]
            .dropna(subset=["canonical_station_id", "canonical_lat", "canonical_lon"])
            .drop_duplicates()
            .copy()
        )
        self.coord_to_station = dict(
            self.canon_unique_df[["coord_key", "canonical_station_id"]].drop_duplicates("coord_key").values
        )
        self.by_name = {k: v.reset_index(drop=True) for k, v in self.canon_unique_df.groupby("normalized_name")}
        self.canon_station_centers = (
            self.canon_unique_df[["canonical_station_id", "canonical_lat", "canonical_lon"]]
            .drop_duplicates("canonical_station_id")
            .reset_index(drop=True)
        )
        self.center_lats = self.canon_station_centers["canonical_lat"].to_numpy()
        self.center_lons = self.canon_station_centers["canonical_lon"].to_numpy()

    @staticmethod
    def _normalize_station_name(text: str) -> str:
        if text is None:
            return ""
        text = str(text).strip().lower()
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s*/\s*", " / ", text)
        text = re.sub(r"\s*-\s*", "-", text)
        return text.strip()

    @staticmethod
    def _haversine_km_vec(lat1, lon1, lat2, lon2):
        r = 6371.0
        lat1_rad = np.radians(lat1)
        lon1_rad = np.radians(lon1)
        lat2_rad = np.radians(lat2)
        lon2_rad = np.radians(lon2)
        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
        return 2 * r * np.arcsin(np.sqrt(a))

    def resolve(self, raw_name: str, lat: float, lon: float) -> dict:
        """
        Resolve using name+coords. Returns dict:
        {"canonical_station_id": ..., "match_method": ..., "match_distance_km": ...}
        """
        normalized_name = self._normalize_station_name(raw_name)
        # guard against NaN coords
        try:
            coord_key = f"{float(lat):.6f},{float(lon):.6f}"
        except Exception:
            coord_key = None

        if coord_key is not None:
            station_id = self.coord_to_station.get(coord_key)
            if station_id is not None:
                return {"canonical_station_id": station_id, "match_method": "exact_coord_key", "match_distance_km": 0.0}

        if normalized_name and normalized_name in self.by_name:
            cand = self.by_name[normalized_name]
            try:
                dists = self._haversine_km_vec(
                    float(lat), float(lon), cand["canonical_lat"].to_numpy(), cand["canonical_lon"].to_numpy()
                )
                min_pos = int(np.argmin(dists))
                min_dist = float(dists[min_pos])
                if min_dist <= self.max_name_coord_km:
                    return {
                        "canonical_station_id": cand.iloc[min_pos]["canonical_station_id"],
                        "match_method": "exact_normalized_name_nearest_coord",
                        "match_distance_km": min_dist,
                    }
            except Exception:
                pass

        try:
            dists = self._haversine_km_vec(float(lat), float(lon), self.center_lats, self.center_lons)
            min_pos = int(np.argmin(dists))
            min_dist = float(dists[min_pos])
            if min_dist <= self.max_nearest_km:
                return {
                    "canonical_station_id": self.canon_station_centers.iloc[min_pos]["canonical_station_id"],
                    "match_method": "nearest_canonical_within_radius",
                    "match_distance_km": min_dist,
                }
        except Exception:
            pass

        return {"canonical_station_id": None, "match_method": "unmatched", "match_distance_km": np.nan}

    def resolve_by_coord(self, lat: float, lon: float) -> dict:
        """
        Convenience wrapper to resolve using only coordinates.
        """
        return self.resolve("", lat, lon)


# --- instantiate resolver (requires canonical_mapping_df) ---
resolver = CanonicalStationResolver(canonical_mapping_df)

# --- Map live stations (GBFS merged info+status) to canonical ids with metadata ---
resolved_records = []
for _, row in live_stations_df.iterrows():
    lat = row.get("lat")
    lon = row.get("lon")
    name = row.get("name", "")
    if pd.isna(lat) or pd.isna(lon):
        match = {"canonical_station_id": None, "match_method": "missing_coords", "match_distance_km": np.nan}
    else:
        match = resolver.resolve(name if pd.notna(name) else "", lat, lon)
    resolved_records.append({
        "station_id": str(row.get("station_id")),
        "name": row.get("name"),
        # some GBFS feeds use 'capacity' in info_df; guard if missing
        "capacity": row.get("capacity") if "capacity" in row.index else np.nan,
        "lat": lat,
        "lon": lon,
        "canonical_station_id": match.get("canonical_station_id"),
        "match_method": match.get("match_method"),
        "match_distance_km": match.get("match_distance_km"),
    })

live_to_canonical_df = pd.DataFrame(resolved_records)

# --- Summary of match methods ---
match_breakdown = live_to_canonical_df["match_method"].value_counts(dropna=False).rename_axis("match_method").reset_index(name="count")
print("Live-to-canonical match breakdown:")
display(match_breakdown)

# --- Build canonical_capacity_df from live mappings (use capacity where available) ---
# rename capacity column to avoid clobbering other 'capacity' columns in top10
canonical_capacity_df = (
    live_to_canonical_df.dropna(subset=["canonical_station_id", "capacity"])
    .groupby("canonical_station_id", as_index=False)
    .agg(capacity_mapped=("capacity", "max"), mapped_live_station_count=("station_id", "nunique"))
)

# --- Ensure top10 exists; if not, compute it from rides_df ---
if 'top10' not in globals():
    # compute connectivity and top10 as fallback
    starts = rides_df["start_canonical_station_id"].value_counts(dropna=True).rename("start_count")
    ends = rides_df["end_canonical_station_id"].value_counts(dropna=True).rename("end_count")
    connectivity_df = (
        pd.concat([starts, ends], axis=1).fillna(0)
        .assign(total_traffic=lambda d: d["start_count"] + d["end_count"])
        .reset_index().rename(columns={"index":"canonical_station_id"})
    )
    pair_df = rides_df.dropna(subset=["start_canonical_station_id","end_canonical_station_id"])[
        ["start_canonical_station_id","end_canonical_station_id"]
    ].drop_duplicates()
    partners = pd.concat([
        pair_df.rename(columns={"start_canonical_station_id":"canonical_station_id","end_canonical_station_id":"partner"}),
        pair_df.rename(columns={"end_canonical_station_id":"canonical_station_id","start_canonical_station_id":"partner"})
    ])
    partner_counts = partners.groupby("canonical_station_id")["partner"].nunique().rename("unique_partners").reset_index()
    top10 = connectivity_df.merge(partner_counts, on="canonical_station_id", how="left").fillna({"unique_partners":0})
    top10 = top10.sort_values("total_traffic", ascending=False).head(10).reset_index(drop=True)

# --- Merge mapped capacities into top10 safely ---
top10 = top10.merge(canonical_capacity_df, on="canonical_station_id", how="left")

# fill missing mapped_live_station_count with 0 and ensure integer type
if "mapped_live_station_count" in top10.columns:
    top10["mapped_live_station_count"] = top10["mapped_live_station_count"].fillna(0).astype(int)
else:
    top10["mapped_live_station_count"] = 0

# If top10 already had a 'capacity' column from canonical_mapping_df, keep it as 'capacity_static'
if "capacity" in top10.columns:
    top10 = top10.rename(columns={"capacity": "capacity_static"})

# Decide final capacity column: prefer mapped capacity, then static capacity, else NaN
def choose_capacity(row):
    if pd.notna(row.get("capacity_mapped")):
        return row["capacity_mapped"], "from_live_mapping"
    if pd.notna(row.get("capacity_static")):
        return row["capacity_static"], "from_canonical_mapping"
    return np.nan, "no_capacity_found"

cap_choice = top10.apply(lambda r: choose_capacity(r), axis=1)
top10["capacity_filled"], top10["capacity_fill_method"] = zip(*cap_choice)

# --- Safe display: pick only columns that exist to avoid KeyError ---
display_cols = []
for c in ["canonical_station_id", "total_traffic", "unique_partners", "canonical_lat", "canonical_lon"]:
    if c in top10.columns:
        display_cols.append(c)
# capacity-related columns
for c in ["capacity_filled", "capacity_mapped", "capacity_static", "capacity_fill_method", "mapped_live_station_count"]:
    if c in top10.columns:
        display_cols.append(c)

print("Top10 with capacity inference and mapping info")
display(top10[display_cols].copy())

Live-to-canonical match breakdown:


,match_method,count
0,exact_coord_key,300
1,exact_normalized_name_nearest_coord,12
2,nearest_canonical_within_radius,2


Top10 with capacity inference and mapping info


,canonical_station_id,total_traffic,unique_partners,capacity_filled,capacity_mapped,capacity_fill_method,mapped_live_station_count
0,STN_0001,474102.0,1125,23.0,23.0,from_live_mapping,1
1,STN_0002,430875.0,1129,39.0,39.0,from_live_mapping,1
2,STN_0003,326292.0,1131,37.0,37.0,from_live_mapping,1
3,STN_0004,293884.0,1087,23.0,23.0,from_live_mapping,1
4,STN_0005,273819.0,1082,31.0,31.0,from_live_mapping,1
5,STN_0006,265097.0,1144,81.0,81.0,from_live_mapping,1
6,STN_0007,261531.0,1110,49.0,49.0,from_live_mapping,1
7,STN_0008,247833.0,1031,NaN,NaN,no_capacity_found,0
8,STN_0009,242268.0,1108,19.0,19.0,from_live_mapping,1
9,STN_0010,241496.0,1106,32.0,32.0,from_live_mapping,1


In [9]:


# --- helper haversine (km) ---
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad)*np.cos(lat2_rad)*np.sin(dlon/2)**2
    return 2 * r * np.arcsin(np.sqrt(a))

# --- 1. Build connectivity metric from rides_df ---
# Option A: undirected traffic = starts + ends counts per canonical id
# Ensure canonical id columns exist
if "start_canonical_station_id" not in rides_df.columns or "end_canonical_station_id" not in rides_df.columns:
    raise ValueError("rides_df missing canonical station id columns")

starts = rides_df["start_canonical_station_id"].value_counts(dropna=True).rename("start_count")
ends = rides_df["end_canonical_station_id"].value_counts(dropna=True).rename("end_count")

connectivity_df = (
    pd.concat([starts, ends], axis=1).fillna(0)
    .assign(total_traffic=lambda d: d["start_count"] + d["end_count"])
    .reset_index()
    .rename(columns={"index": "canonical_station_id"})
)

# Optionally compute directed in/out degree or unique partners (connectivity graph degree)
# Unique partner count (how many distinct other canonical stations connect to this one)
pair_df = rides_df.dropna(subset=["start_canonical_station_id", "end_canonical_station_id"])[
    ["start_canonical_station_id", "end_canonical_station_id"]
].drop_duplicates()
# undirected partner counts
partners = pd.concat([
    pair_df.rename(columns={"start_canonical_station_id":"canonical_station_id", "end_canonical_station_id":"partner"}),
    pair_df.rename(columns={"end_canonical_station_id":"canonical_station_id", "start_canonical_station_id":"partner"})
])
partner_counts = partners.groupby("canonical_station_id")["partner"].nunique().rename("unique_partners").reset_index()

connectivity_df = connectivity_df.merge(partner_counts, on="canonical_station_id", how="left").fillna({"unique_partners":0})

# --- 2. Top 10 by chosen metric ---
# Choose metric: 'total_traffic' or 'unique_partners'
metric = "total_traffic"   # change to "unique_partners" if you prefer connectivity degree
top10 = connectivity_df.sort_values(metric, ascending=False).head(10).reset_index(drop=True)
print("Top 10 canonical stations by", metric)
display(top10)

# --- 3. Attach canonical coordinates from canonical_mapping_df ---
# canonical_mapping_df must contain canonical_station_id, canonical_lat, canonical_lon
required = {"canonical_station_id", "canonical_lat", "canonical_lon"}
if not required.issubset(set(canonical_mapping_df.columns)):
    raise ValueError(f"canonical_mapping_df missing required columns: {required - set(canonical_mapping_df.columns)}")

canon_coords = canonical_mapping_df[["canonical_station_id", "canonical_lat", "canonical_lon"]].drop_duplicates("canonical_station_id")
top10 = top10.merge(canon_coords, on="canonical_station_id", how="left")

# --- 4. Pairwise distances between top 10 ---
pairs = []
for (i, r1), (j, r2) in combinations(top10.iterrows(), 2):
    d_km = haversine_km(r1["canonical_lat"], r1["canonical_lon"], r2["canonical_lat"], r2["canonical_lon"])
    pairs.append({
        "station_a": r1["canonical_station_id"],
        "station_b": r2["canonical_station_id"],
        "distance_km": float(d_km)
    })
pairs_df = pd.DataFrame(pairs).sort_values("distance_km").reset_index(drop=True)
print("Pairwise distances among top 10 (km):")
display(pairs_df)

# --- 5. Verify canonical ids exist in status_df ---
# status_df has 'station_id' column representing live station ids; if your status_df uses canonical ids, adapt accordingly.
# If status_df.station_id corresponds to canonical_station_id, check membership directly:
if "station_id" in status_df.columns:
    # If station_id is numeric but canonical ids are strings, coerce to str for comparison
    status_ids = status_df["station_id"].astype(str).unique()
    top10["in_status_df"] = top10["canonical_station_id"].astype(str).isin(status_ids)
else:
    top10["in_status_df"] = False

print("Top10 membership in status_df:")
display(top10[["canonical_station_id", metric, "unique_partners", "canonical_lat", "canonical_lon", "in_status_df"]])

# --- 6. If status_df uses live station ids and you need to map live->canonical, use CanonicalStationResolver ---
# Use your resolver instance (resolver) and live_stations_df (must have name, lat, lon, station_id, capacity)
if 'resolver' in globals() and 'live_stations_df' in globals():
    resolved_records = []
    for _, row in live_stations_df.iterrows():
        r = resolver.resolve(row["name"], row["lat"], row["lon"])
        resolved_records.append({
            "station_id": row["station_id"],
            "name": row["name"],
            "capacity": row.get("capacity", np.nan),
            "lat": row["lat"],
            "lon": row["lon"],
            "canonical_station_id": r["canonical_station_id"],
            "match_method": r["match_method"],
            "match_distance_km": r["match_distance_km"],
        })
    live_to_canonical_df = pd.DataFrame(resolved_records)
    # summary
    match_breakdown = live_to_canonical_df["match_method"].value_counts(dropna=False).rename_axis("match_method").reset_index(name="count")
    print("Live-to-canonical match breakdown:")
    display(match_breakdown)
    # canonical capacities inferred from live stations
    canonical_capacity_df = (
        live_to_canonical_df.dropna(subset=["canonical_station_id", "capacity"])
        .groupby("canonical_station_id", as_index=False)
        .agg(capacity=("capacity", "max"), mapped_live_station_count=("station_id", "nunique"))
    )
    # join capacities to top10
    top10 = top10.merge(canonical_capacity_df, on="canonical_station_id", how="left")
    print("Top10 with inferred capacity and mapping info:")
    display(top10[["canonical_station_id", metric, "capacity", "mapped_live_station_count", "canonical_lat", "canonical_lon"]])
else:
    print("Resolver or live_stations_df not found in globals; skipping live->canonical mapping step.")

# --- 7. Report unmatched top stations ---
unmatched = top10[top10["canonical_lat"].isna() | top10["canonical_lon"].isna()]
if not unmatched.empty:
    print("Warning: some top canonical stations lack coordinates in canonical_mapping_df:")
    display(unmatched[["canonical_station_id", metric]])

Top 10 canonical stations by total_traffic


,canonical_station_id,start_count,end_count,total_traffic,unique_partners
0,STN_0001,249555.0,224547.0,474102.0,1125
1,STN_0002,229304.0,201571.0,430875.0,1129
2,STN_0003,180266.0,146026.0,326292.0,1131
3,STN_0004,152579.0,141305.0,293884.0,1087
4,STN_0005,137603.0,136216.0,273819.0,1082
5,STN_0006,106102.0,158995.0,265097.0,1144
6,STN_0007,127581.0,133950.0,261531.0,1110
7,STN_0008,126015.0,121818.0,247833.0,1031
8,STN_0009,122592.0,119676.0,242268.0,1108
9,STN_0010,107896.0,133600.0,241496.0,1106


Pairwise distances among top 10 (km):


,station_a,station_b,distance_km
0,STN_0007,STN_0009,0.510515
1,STN_0003,STN_0005,0.692051
2,STN_0001,STN_0002,0.693015
3,STN_0001,STN_0003,0.697196
4,STN_0005,STN_0008,0.727894
5,STN_0002,STN_0003,0.884400
6,STN_0001,STN_0005,0.937035
7,STN_0001,STN_0008,0.991646
8,STN_0002,STN_0004,1.027095
9,STN_0001,STN_0004,1.128362


Top10 membership in status_df:


,canonical_station_id,total_traffic,unique_partners,canonical_lat,canonical_lon,in_status_df
0,STN_0001,474102.0,1125,45.524353,-73.581432,False
1,STN_0002,430875.0,1129,45.519410,-73.586850,False
2,STN_0003,326292.0,1131,45.527154,-73.589439,False
3,STN_0004,293884.0,1087,45.515228,-73.575096,False
4,STN_0005,273819.0,1082,45.532449,-73.584770,False
5,STN_0006,265097.0,1144,45.507610,-73.551836,False
6,STN_0007,261531.0,1110,45.500380,-73.575070,False
7,STN_0008,247833.0,1031,45.532218,-73.575431,False
8,STN_0009,242268.0,1108,45.496840,-73.579241,False
9,STN_0010,241496.0,1106,45.523845,-73.552366,False


Live-to-canonical match breakdown:


,match_method,count
0,exact_coord_key,300
1,exact_normalized_name_nearest_coord,12
2,nearest_canonical_within_radius,2


Top10 with inferred capacity and mapping info:


,canonical_station_id,total_traffic,capacity,mapped_live_station_count,canonical_lat,canonical_lon
0,STN_0001,474102.0,23.0,1.0,45.524353,-73.581432
1,STN_0002,430875.0,39.0,1.0,45.519410,-73.586850
2,STN_0003,326292.0,37.0,1.0,45.527154,-73.589439
3,STN_0004,293884.0,23.0,1.0,45.515228,-73.575096
4,STN_0005,273819.0,31.0,1.0,45.532449,-73.584770
5,STN_0006,265097.0,81.0,1.0,45.507610,-73.551836
6,STN_0007,261531.0,49.0,1.0,45.500380,-73.575070
7,STN_0008,247833.0,NaN,NaN,45.532218,-73.575431
8,STN_0009,242268.0,19.0,1.0,45.496840,-73.579241
9,STN_0010,241496.0,32.0,1.0,45.523845,-73.552366


In [10]:
# --- Cell 1: list live station_id and name for the two match methods ---
methods = ["exact_normalized_name_nearest_coord", "nearest_canonical_within_radius"]

for m in methods:
    print(f"\nMatch method: {m}")
    if 'live_to_canonical_df' not in globals():
        print("Error: live_to_canonical_df not found in globals()")
        break
    subset = live_to_canonical_df[live_to_canonical_df["match_method"] == m].copy()
    if subset.empty:
        print("  (no rows)")
        continue
    # ensure columns exist
    cols = []
    if "station_id" in subset.columns:
        cols.append("station_id")
    if "name" in subset.columns:
        cols.append("name")
    if "canonical_station_id" in subset.columns:
        cols.append("canonical_station_id")
    if "match_distance_km" in subset.columns:
        cols.append("match_distance_km")
    # display concise table
    display(subset[cols].reset_index(drop=True))


Match method: exact_normalized_name_nearest_coord


,station_id,name,canonical_station_id,match_distance_km
0,3,Clark / Ontario,STN_0346,0.010890
1,19,Métro Sherbrooke (de Rigaud / Berri),STN_0020,0.005808
2,24,Notre-Dame / St-Gabriel,STN_0481,0.004048
3,26,de Maisonneuve / Mansfield (sud),STN_0125,0.028747
4,34,Viger / Chenneville,STN_0320,0.008555
5,47,Garnier / Beaubien,STN_0338,0.022812
6,49,Robert-Bourassa / St-Maurice,STN_0316,0.025869
7,580,City Councillors / du President-Kennedy,STN_0397,0.012580
8,593,du President-Kennedy / Union,STN_0243,0.020449
9,610,Mansfield / Ste-Catherine,STN_0317,0.010925



Match method: nearest_canonical_within_radius


,station_id,name,canonical_station_id,match_distance_km
0,52,Place des Festival (du Président-Kennedy / Ble...,STN_0101,0.005149
1,157,Chambord / du Mont-Royal,STN_0074,0.049385


In [11]:
# --- Cell 2: show canonical STN_0008 details and any live stations mapped to it ---
target_cid = "STN_0008"

# 1) canonical mapping row(s)
print("Canonical mapping rows for", target_cid)
if 'canonical_mapping_df' in globals():
    cm = canonical_mapping_df[canonical_mapping_df["canonical_station_id"].astype(str).str.strip() == str(target_cid)]
    if cm.empty:
        print("  (no canonical_mapping_df row found for this id)")
    else:
        display(cm.reset_index(drop=True))
else:
    print("  canonical_mapping_df not found in globals()")

# 2) top10 row (if present)
print("\nTop10 row for", target_cid)
if 'top10' in globals():
    t = top10[top10["canonical_station_id"].astype(str).str.strip() == str(target_cid)]
    if t.empty:
        print("  (not present in top10 DataFrame)")
    else:
        display(t.reset_index(drop=True))
else:
    print("  top10 not found in globals()")

# 3) live stations that mapped to this canonical id
print("\nLive stations mapped to", target_cid)
if 'live_to_canonical_df' in globals():
    mapped = live_to_canonical_df[live_to_canonical_df["canonical_station_id"].astype(str).str.strip() == str(target_cid)].copy()
    if mapped.empty:
        print("  (no live stations mapped to this canonical id)")
    else:
        # show useful columns and sort by distance if available
        show_cols = []
        for c in ["station_id", "name", "capacity", "lat", "lon", "match_method", "match_distance_km"]:
            if c in mapped.columns:
                show_cols.append(c)
        if "match_distance_km" in mapped.columns:
            mapped = mapped.sort_values("match_distance_km")
        display(mapped[show_cols].reset_index(drop=True))
else:
    print("  live_to_canonical_df not found in globals()")

# 4) If no live mapping found, show nearest live station(s) by distance from canonical center (helpful to fill capacity)
if ('canonical_mapping_df' in globals()) and (('live_to_canonical_df' in globals() and mapped.empty) or 'live_to_canonical_df' not in globals()):
    cm_row = canonical_mapping_df[canonical_mapping_df["canonical_station_id"].astype(str).str.strip() == str(target_cid)]
    if not cm_row.empty and "canonical_lat" in cm_row.columns and "canonical_lon" in cm_row.columns:
        lat0 = float(cm_row.iloc[0]["canonical_lat"])
        lon0 = float(cm_row.iloc[0]["canonical_lon"])
        # build live source to search: prefer live_to_canonical_df (info_df merged) else info_df
        if 'live_to_canonical_df' in globals() and not live_to_canonical_df.empty:
            search_df = live_to_canonical_df.dropna(subset=["lat","lon"]).copy()
        elif 'info_df' in globals():
            search_df = info_df.rename(columns={"lat":"lat","lon":"lon"})  # ensure columns exist
        else:
            search_df = None

        if search_df is None or search_df.empty:
            print("\nNo live/info coordinates available to compute nearest live stations.")
        else:
            # compute haversine distances
            import numpy as _np
            def _haversine_km(lat1, lon1, lat2, lon2):
                r = 6371.0
                lat1_rad, lon1_rad = _np.radians(lat1), _np.radians(lon1)
                lat2_rad, lon2_rad = _np.radians(lat2), _np.radians(lon2)
                dlat = lat2_rad - lat1_rad
                dlon = lon2_rad - lon1_rad
                a = _np.sin(dlat/2)**2 + _np.cos(lat1_rad)*_np.cos(lat2_rad)*_np.sin(dlon/2)**2
                return 2 * r * _np.arcsin(_np.sqrt(a))
            search_df = search_df.assign(
                dist_km = _haversine_km(lat0, lon0, search_df["lat"].to_numpy(), search_df["lon"].to_numpy())
            )
            nearest = search_df.sort_values("dist_km").head(10)
            print("\nNearest live/info stations to canonical STN_0008 (top 10):")
            display(nearest[["station_id","name","lat","lon","dist_km", "capacity"]].reset_index(drop=True))
    else:
        print("\nCanonical mapping row lacks coordinates; cannot compute nearest live stations.")

Canonical mapping rows for STN_0008


,station_key,coord_key,normalized_name,canonical_station_id,canonical_coord_key,canonical_lat,canonical_lon,cluster_id,cluster_size,observed_trip_count,first_year_seen,last_year_seen
0,"45.532218,-73.575431|marquette / du mont-royal","45.532218,-73.575431",marquette / du mont-royal,STN_0008,"45.532218,-73.575431",45.532218,-73.575431,cluster_000872,1,247833,2024,2025



Top10 row for STN_0008


,canonical_station_id,start_count,end_count,total_traffic,unique_partners,canonical_lat,canonical_lon,in_status_df,capacity,mapped_live_station_count
0,STN_0008,126015.0,121818.0,247833.0,1031,45.532218,-73.575431,False,NaN,NaN



Live stations mapped to STN_0008
  (no live stations mapped to this canonical id)

Nearest live/info stations to canonical STN_0008 (top 10):


,station_id,name,lat,lon,dist_km,capacity
0,161,Marquette / du Mont-Royal (sud),45.532077,-73.575143,0.027350,23
1,912,Garnier / du Mont-Royal,45.530756,-73.576176,0.172623,23
2,544,Cartier / Marie-Anne,45.532804,-73.571902,0.282532,27
3,157,Chambord / du Mont-Royal,45.529712,-73.577218,0.311547,11
4,916,de Bordeaux / Gilford,45.536090,-73.576168,0.434402,23
5,141,Émile-Duployé / Rachel,45.529707,-73.569981,0.508097,25
6,138,Calixa-Lavallée / Rachel (ouest),45.527829,-73.571958,0.558015,37
7,151,Parthenais / du Mont-Royal,45.536404,-73.571413,0.560869,23
8,169,du Mont-Royal / Christophe-Colomb,45.527954,-73.579362,0.564438,11
9,957,Calixa-Lavallée / Rachel,45.527783,-73.571674,0.573431,6


Canonical station ID STN_0008 should map to station_id 161, which is also mapped to canonical station id STN_0158

In [12]:

# --- configure output directory ---
output_dir = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# --- mapping of variable name -> filename ---
to_export = {
    "connectivity_df": output_dir / "connectivity_by_canonical.csv",
    "top10": output_dir / "top10_canonical_stations.csv",
    "pairs_df": output_dir / "top10_pairwise_distances_km.csv",
    "live_to_canonical_df": output_dir / "live_to_canonical_mapping.csv",
    "canonical_capacity_df": output_dir / "canonical_capacity_from_live.csv",
    "match_breakdown": output_dir / "live_to_canonical_match_breakdown.csv",
    "unmatched": output_dir / "top10_unmatched_coordinates.csv",
}

written = []
missing = []

# --- helper to write a DataFrame safely ---
def safe_to_csv(var_name, path):
    if var_name not in globals():
        missing.append(var_name)
        return
    df = globals()[var_name]
    if df is None:
        missing.append(var_name)
        return
    if not isinstance(df, pd.DataFrame):
        # try to coerce simple structures to DataFrame
        try:
            df = pd.DataFrame(df)
        except Exception:
            missing.append(var_name)
            return
    # ensure no problematic index columns and consistent column types
    try:
        df.to_csv(path, index=False)
        written.append((var_name, str(path), len(df)))
    except Exception as e:
        # attempt a safe write by converting object columns to strings
        try:
            df_safe = df.copy()
            for c in df_safe.columns:
                if df_safe[c].dtype == object:
                    df_safe[c] = df_safe[c].astype(str)
            df_safe.to_csv(path, index=False)
            written.append((var_name, str(path), len(df_safe)))
        except Exception as e2:
            missing.append(var_name)

# --- run exports ---
for var, path in to_export.items():
    safe_to_csv(var, path)

# --- summary printout ---
print("Export summary:")
if written:
    for name, path, nrows in written:
        print(f"  WROTE: {name} -> {path} ({nrows:,} rows)")
if missing:
    print("\nMissing or failed to export (not found or not a DataFrame):")
    for name in missing:
        print(f"  - {name}")

# --- quick checks: show where STN_0008 details were written if present ---
target_cid = "STN_0008"
if "top10" in globals():
    t = top10[top10["canonical_station_id"].astype(str).str.strip() == target_cid]
    if not t.empty:
        print(f"\nTop10 row for {target_cid} saved (if top10 exported).")
if "live_to_canonical_df" in globals():
    mapped = live_to_canonical_df[live_to_canonical_df["canonical_station_id"].astype(str).str.strip() == target_cid]
    if not mapped.empty:
        print(f"Live stations mapped to {target_cid} saved in live_to_canonical_mapping.csv.")

Export summary:
  WROTE: connectivity_df -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\connectivity_by_canonical.csv (1,422 rows)
  WROTE: top10 -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\top10_canonical_stations.csv (10 rows)
  WROTE: pairs_df -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\top10_pairwise_distances_km.csv (45 rows)
  WROTE: live_to_canonical_df -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\live_to_canonical_mapping.csv (314 rows)
  WROTE: canonical_capacity_df -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\canonical_capacity_from_live.csv (314 rows)
  WROTE: match_breakdown -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\live_to_canonical_match_breakdown.csv (3 rows)
  WROTE: unmatched -> C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\top10_unmatched_coordinate

In [13]:
from pathlib import Path
import numpy as np
import pandas as pd

# --- parameters ---
OUTPUT_DIR = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NO_CAP_CSV = OUTPUT_DIR / "no_capacity_canonical_stations.csv"
NEAREST_CANDIDATES_CSV = OUTPUT_DIR / "no_capacity_nearest_live_candidates.csv"
TOP_N_NEAREST = 10
MAX_NEAREST_KM = 0.5  # optional cap; set to None to keep all distances

# --- helper haversine (km) ---
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad)*np.cos(lat2_rad)*np.sin(dlon/2)**2
    return 2 * r * np.arcsin(np.sqrt(a))

# --- 0. Preconditions and fallbacks ---
# Ensure canonical_mapping_df exists
if "canonical_mapping_df" not in globals():
    raise RuntimeError("canonical_mapping_df not found in globals()")

# Ensure we have a DataFrame with capacity_fill_method or capacity_filled; try to build from top10 if needed
if "top10" not in globals():
    # attempt to compute top10 from connectivity_df if available
    if "connectivity_df" in globals():
        top10 = connectivity_df.sort_values("total_traffic", ascending=False).head(10).reset_index(drop=True)
        # attach coords if possible
        if set(["canonical_lat","canonical_lon"]).issubset(set(canonical_mapping_df.columns)):
            top10 = top10.merge(
                canonical_mapping_df[["canonical_station_id","canonical_lat","canonical_lon"]].drop_duplicates("canonical_station_id"),
                on="canonical_station_id", how="left"
            )
    else:
        raise RuntimeError("top10 not found and connectivity_df not available to compute it")

# If capacity_filled / capacity_fill_method not present, try to compute using earlier logic
if "capacity_filled" not in top10.columns or "capacity_fill_method" not in top10.columns:
    # attempt to attach any mapped capacity from live mapping if available
    if "live_to_canonical_df" in globals():
        cap_df = (
            live_to_canonical_df.dropna(subset=["canonical_station_id","capacity"])
            .groupby("canonical_station_id", as_index=False)
            .agg(capacity_mapped=("capacity","max"), mapped_live_station_count=("station_id","nunique"))
        )
        top10 = top10.merge(cap_df, on="canonical_station_id", how="left")
        # fallback to any static capacity in canonical_mapping_df or info_df
        if "capacity" in canonical_mapping_df.columns:
            top10 = top10.merge(canonical_mapping_df[["canonical_station_id","capacity"]].drop_duplicates("canonical_station_id"),
                                on="canonical_station_id", how="left", suffixes=("","_static"))
            top10["capacity_filled"] = top10.get("capacity_mapped").fillna(top10.get("capacity"))
            top10["capacity_fill_method"] = np.where(top10["capacity_mapped"].notna(), "from_live_mapping",
                                                     np.where(top10["capacity"].notna(), "from_canonical_mapping", "no_capacity_found"))
        else:
            top10["capacity_filled"] = top10.get("capacity_mapped")
            top10["capacity_fill_method"] = np.where(top10["capacity_mapped"].notna(), "from_live_mapping", "no_capacity_found")
    else:
        # no live mapping; try info_df fallback
        if "info_df" in globals():
            info_small = info_df.copy()
            if "station_id" in info_small.columns:
                info_small["station_id"] = info_small["station_id"].astype(str)
            # map info_df to canonical via resolver if available
            if 'resolver' in globals():
                info_res = []
                for _, r in info_small.iterrows():
                    lat = r.get("lat"); lon = r.get("lon")
                    if pd.isna(lat) or pd.isna(lon):
                        cid = None
                    else:
                        cid = resolver.resolve_by_coord(lat, lon).get("canonical_station_id")
                    info_res.append({"station_id": r.get("station_id"), "canonical_station_id": cid, "capacity_info": r.get("capacity")})
                info_res_df = pd.DataFrame(info_res)
                info_cap_by_canon = info_res_df.dropna(subset=["canonical_station_id","capacity_info"]).groupby("canonical_station_id", as_index=False).agg(capacity_info_max=("capacity_info","max"))
                top10 = top10.merge(info_cap_by_canon, on="canonical_station_id", how="left")
                top10["capacity_filled"] = top10.get("capacity_info_max")
                top10["capacity_fill_method"] = np.where(top10["capacity_info_max"].notna(), "from_info_df_fallback", "no_capacity_found")
            else:
                top10["capacity_filled"] = np.nan
                top10["capacity_fill_method"] = "no_capacity_found"
        else:
            top10["capacity_filled"] = np.nan
            top10["capacity_fill_method"] = "no_capacity_found"

# --- 1. Identify canonical stations with no capacity found ---
no_cap_mask = (top10["capacity_filled"].isna()) | (top10.get("capacity_fill_method") == "no_capacity_found")
no_cap_df = top10[no_cap_mask].copy()
# include canonical coords and total_traffic
cols_keep = [c for c in ["canonical_station_id","total_traffic","unique_partners","canonical_lat","canonical_lon","capacity_filled","capacity_fill_method"] if c in no_cap_df.columns]
no_cap_export = no_cap_df[cols_keep].reset_index(drop=True)

# write the no-capacity canonical list
no_cap_export.to_csv(NO_CAP_CSV, index=False)

# --- 2. For each no-cap canonical, find nearest live/info stations and include capacity if available ---
candidates_rows = []
# build search source: prefer live_to_canonical_df (which contains capacity from GBFS), else info_df
if 'live_to_canonical_df' in globals() and not live_to_canonical_df.empty:
    search_df = live_to_canonical_df.dropna(subset=["lat","lon"]).copy()
    # ensure capacity column exists
    if "capacity" not in search_df.columns and "capacity_mapped" in search_df.columns:
        search_df = search_df.rename(columns={"capacity_mapped":"capacity"})
elif 'info_df' in globals() and not info_df.empty:
    search_df = info_df.copy()
    # normalize common columns
    if "station_id" in search_df.columns:
        search_df["station_id"] = search_df["station_id"].astype(str)
else:
    search_df = pd.DataFrame()  # empty

for _, row in no_cap_export.iterrows():
    cid = row["canonical_station_id"]
    lat0 = row.get("canonical_lat")
    lon0 = row.get("canonical_lon")
    if pd.isna(lat0) or pd.isna(lon0) or search_df.empty:
        # record an empty candidate row to indicate no search possible
        candidates_rows.append({
            "canonical_station_id": cid,
            "candidate_rank": None,
            "station_id": None,
            "name": None,
            "lat": None,
            "lon": None,
            "distance_km": None,
            "capacity": None
        })
        continue

    # compute distances to all search_df rows
    coords_lat = search_df["lat"].to_numpy()
    coords_lon = search_df["lon"].to_numpy()
    dists = haversine_km(lat0, lon0, coords_lat, coords_lon)
    search_df = search_df.assign(_dist_km = dists)
    # optionally filter by MAX_NEAREST_KM
    if MAX_NEAREST_KM is not None:
        cand_df = search_df[search_df["_dist_km"] <= MAX_NEAREST_KM].copy()
    else:
        cand_df = search_df.copy()
    if cand_df.empty:
        # still include the nearest even if beyond MAX_NEAREST_KM
        nearest_idx = int(np.argmin(dists))
        cand_df = search_df.iloc[[nearest_idx]].copy()

    cand_df = cand_df.sort_values("_dist_km").head(TOP_N_NEAREST)
    # pick display columns and capacity if present
    for rank, (_, c) in enumerate(cand_df.iterrows(), start=1):
        candidates_rows.append({
            "canonical_station_id": cid,
            "candidate_rank": rank,
            "station_id": c.get("station_id"),
            "name": c.get("name"),
            "lat": c.get("lat"),
            "lon": c.get("lon"),
            "distance_km": float(c.get("_dist_km")) if pd.notna(c.get("_dist_km")) else None,
            "capacity": c.get("capacity") if "capacity" in c.index else None,
            "match_method": c.get("match_method") if "match_method" in c.index else None,
            "match_distance_km": c.get("match_distance_km") if "match_distance_km" in c.index else None
        })

# build DataFrame and export
candidates_df = pd.DataFrame(candidates_rows)
candidates_df.to_csv(NEAREST_CANDIDATES_CSV, index=False)

# --- 3. Report back what was written ---
written = []
if NO_CAP_CSV.exists():
    written.append((str(NO_CAP_CSV), len(no_cap_export)))
if NEAREST_CANDIDATES_CSV.exists():
    written.append((str(NEAREST_CANDIDATES_CSV), len(candidates_df)))

print("Exported files:")
for path, nrows in written:
    print(f" - {path} ({nrows} rows)")
if not written:
    print("No files were written; check preconditions and variables in the notebook.")

Exported files:
 - C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\no_capacity_canonical_stations.csv (1 rows)
 - C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\no_capacity_nearest_live_candidates.csv (5 rows)


Re-test and adding connectivity (traffic for routes)

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product, combinations

# --- output directory ---
out_dir = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# --- Preconditions checks ---
if "rides_df" not in globals():
    raise RuntimeError("rides_df not found in globals()")
if "top10" not in globals():
    raise RuntimeError("top10 not found in globals()")
if "canonical_mapping_df" not in globals():
    raise RuntimeError("canonical_mapping_df not found in globals()")

# canonical id column names expected in rides_df
start_col = "start_canonical_station_id"
end_col = "end_canonical_station_id"
if start_col not in rides_df.columns or end_col not in rides_df.columns:
    raise ValueError(f"rides_df must contain columns {start_col} and {end_col}")

# ensure top10 canonical ids list
top10_cids = list(top10["canonical_station_id"].astype(str).tolist())
print("Top10 canonical ids:", top10_cids)

# --- 1. Filter rides to those involving top10 as origin or destination ---
mask = rides_df[start_col].astype(str).isin(top10_cids) | rides_df[end_col].astype(str).isin(top10_cids)
rides_top10 = rides_df[mask].copy()
print("Rides involving top10 stations rows:", len(rides_top10))

# --- 2. Directed pair counts among top10 (origin -> destination) ---
# keep only rides where both origin and destination are in top10 for pairwise matrix
both_mask = rides_top10[start_col].astype(str).isin(top10_cids) & rides_top10[end_col].astype(str).isin(top10_cids)
rides_top10_pairs = rides_top10[both_mask].copy()
print("Rides with both endpoints in top10:", len(rides_top10_pairs))

directed_counts = (
    rides_top10_pairs
    .groupby([start_col, end_col], dropna=False)
    .size()
    .rename("ride_count")
    .reset_index()
)
# pivot to matrix form (rows = origin, cols = destination)
directed_matrix = directed_counts.pivot_table(index=start_col, columns=end_col, values="ride_count", fill_value=0)

# ensure all top10 ids appear as rows and cols
directed_matrix = directed_matrix.reindex(index=top10_cids, columns=top10_cids, fill_value=0)

# --- 3. Undirected pair counts (treat A->B and B->A as same pair) ---
# create canonical unordered pair key
def unordered_pair(a, b):
    a, b = str(a), str(b)
    return tuple(sorted([a, b]))

# aggregate directed_counts into undirected
directed_counts["pair_key"] = directed_counts.apply(lambda r: unordered_pair(r[start_col], r[end_col]), axis=1)
undirected_agg = (
    directed_counts.groupby("pair_key")["ride_count"]
    .sum()
    .reset_index()
)
# expand pair_key into station_a, station_b
undirected_agg[["station_a", "station_b"]] = pd.DataFrame(undirected_agg["pair_key"].tolist(), index=undirected_agg.index)
undirected_agg = undirected_agg[["station_a", "station_b", "ride_count"]].sort_values("ride_count", ascending=False).reset_index(drop=True)

# --- 4. Top 10 most frequent directed pairs and undirected pairs ---
top10_directed_pairs = directed_counts.sort_values("ride_count", ascending=False).head(10).reset_index(drop=True)
top10_undirected_pairs = undirected_agg.head(10).copy()

# --- 5. Pairwise distances among top10 canonical centers ---
# get canonical coords for top10
canon_coords = canonical_mapping_df[["canonical_station_id", "canonical_lat", "canonical_lon"]].drop_duplicates("canonical_station_id")
canon_coords = canon_coords.set_index("canonical_station_id").reindex(top10_cids)
if canon_coords["canonical_lat"].isna().any() or canon_coords["canonical_lon"].isna().any():
    print("Warning: some top10 canonical stations lack coordinates in canonical_mapping_df")

pairs = []
for a, b in combinations(top10_cids, 2):
    lat_a, lon_a = canon_coords.loc[a, "canonical_lat"], canon_coords.loc[a, "canonical_lon"]
    lat_b, lon_b = canon_coords.loc[b, "canonical_lat"], canon_coords.loc[b, "canonical_lon"]
    if pd.isna(lat_a) or pd.isna(lon_a) or pd.isna(lat_b) or pd.isna(lon_b):
        dist_km = np.nan
    else:
        dist_km = haversine_km(lat_a, lon_a, lat_b, lon_b)
    pairs.append({"station_a": a, "station_b": b, "distance_km": float(dist_km) if not pd.isna(dist_km) else np.nan})
pairs_df = pd.DataFrame(pairs).sort_values("distance_km").reset_index(drop=True)

# --- 6. Export results to CSV files ---
directed_matrix_csv = out_dir / "top10_directed_connectivity_matrix.csv"
directed_counts_csv = out_dir / "top10_directed_pair_counts.csv"
undirected_pairs_csv = out_dir / "top10_undirected_pair_counts.csv"
top10_directed_csv = out_dir / "top10_directed_pairs_top10.csv"
top10_undirected_csv = out_dir / "top10_undirected_pairs_top10.csv"
pairs_dist_csv = out_dir / "top10_pairwise_distances_km.csv"

# write files
directed_matrix.reset_index().to_csv(directed_matrix_csv, index=False)
directed_counts.to_csv(directed_counts_csv, index=False)
undirected_agg.to_csv(undirected_pairs_csv, index=False)
top10_directed_pairs.to_csv(top10_directed_csv, index=False)
top10_undirected_pairs.to_csv(top10_undirected_csv, index=False)
pairs_df.to_csv(pairs_dist_csv, index=False)

# --- 7. Print short summaries and display key tables ---
print("\nWrote CSVs to", out_dir)
print(f"Directed matrix shape: {directed_matrix.shape} -> {directed_matrix_csv.name}")
print(f"Directed pair counts rows: {len(directed_counts)} -> {directed_counts_csv.name}")
print(f"Undirected pair counts rows: {len(undirected_agg)} -> {undirected_pairs_csv.name}")
print(f"Top 10 directed pairs -> {top10_directed_csv.name}")
print(f"Top 10 undirected pairs -> {top10_undirected_csv.name}")
print(f"Pairwise distances -> {pairs_dist_csv.name}")

# display small samples
print("\nDirected connectivity matrix (sample):")
display(directed_matrix)

print("\nTop 10 directed pairs (origin -> destination, ride_count):")
display(top10_directed_pairs)

print("\nTop 10 undirected pairs (station_a - station_b, ride_count):")
display(top10_undirected_pairs)

print("\nPairwise distances among top10 (km) sample:")
display(pairs_df.head(20))

Top10 canonical ids: ['STN_0001', 'STN_0002', 'STN_0003', 'STN_0004', 'STN_0005', 'STN_0006', 'STN_0007', 'STN_0008', 'STN_0009', 'STN_0010']
Rides involving top10 stations rows: 2910655
Rides with both endpoints in top10: 146542

Wrote CSVs to C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs
Directed matrix shape: (10, 10) -> top10_directed_connectivity_matrix.csv
Directed pair counts rows: 100 -> top10_directed_pair_counts.csv
Undirected pair counts rows: 55 -> top10_undirected_pair_counts.csv
Top 10 directed pairs -> top10_directed_pairs_top10.csv
Top 10 undirected pairs -> top10_undirected_pairs_top10.csv
Pairwise distances -> top10_pairwise_distances_km.csv

Directed connectivity matrix (sample):


end_canonical_station_id,STN_0001,STN_0002,STN_0003,STN_0004,STN_0005,STN_0006,STN_0007,STN_0008,STN_0009,STN_0010
start_canonical_station_id,,,,,,,,,,
STN_0001,6710.0,5903.0,1488.0,1422.0,1216.0,630.0,328.0,7993.0,301.0,728.0
STN_0002,8121.0,4885.0,1813.0,2680.0,1595.0,337.0,922.0,1269.0,824.0,526.0
STN_0003,1412.0,2368.0,3679.0,1051.0,5044.0,304.0,281.0,1370.0,133.0,262.0
STN_0004,1676.0,2283.0,836.0,3776.0,630.0,406.0,982.0,478.0,1119.0,377.0
STN_0005,1136.0,1392.0,4485.0,585.0,2633.0,382.0,211.0,2122.0,103.0,226.0
STN_0006,332.0,185.0,176.0,290.0,152.0,7730.0,293.0,120.0,297.0,671.0
STN_0007,374.0,619.0,198.0,879.0,172.0,606.0,3434.0,142.0,3127.0,552.0
STN_0008,7492.0,1171.0,1222.0,425.0,2381.0,338.0,232.0,4199.0,42.0,623.0
STN_0009,248.0,622.0,153.0,990.0,107.0,551.0,3106.0,33.0,4445.0,316.0



Top 10 directed pairs (origin -> destination, ride_count):


,start_canonical_station_id,end_canonical_station_id,ride_count,pair_key
0,STN_0002,STN_0001,8121,"(STN_0001, STN_0002)"
1,STN_0001,STN_0008,7993,"(STN_0001, STN_0008)"
2,STN_0006,STN_0006,7730,"(STN_0006, STN_0006)"
3,STN_0008,STN_0001,7492,"(STN_0001, STN_0008)"
4,STN_0001,STN_0001,6710,"(STN_0001, STN_0001)"
5,STN_0001,STN_0002,5903,"(STN_0001, STN_0002)"
6,STN_0003,STN_0005,5044,"(STN_0003, STN_0005)"
7,STN_0002,STN_0002,4885,"(STN_0002, STN_0002)"
8,STN_0005,STN_0003,4485,"(STN_0003, STN_0005)"
9,STN_0009,STN_0009,4445,"(STN_0009, STN_0009)"



Top 10 undirected pairs (station_a - station_b, ride_count):


,station_a,station_b,ride_count
0,STN_0001,STN_0008,15485
1,STN_0001,STN_0002,14024
2,STN_0003,STN_0005,9529
3,STN_0006,STN_0006,7730
4,STN_0001,STN_0001,6710
5,STN_0007,STN_0009,6233
6,STN_0002,STN_0004,4963
7,STN_0002,STN_0002,4885
8,STN_0005,STN_0008,4503
9,STN_0009,STN_0009,4445



Pairwise distances among top10 (km) sample:


,station_a,station_b,distance_km
0,STN_0007,STN_0009,0.510515
1,STN_0003,STN_0005,0.692051
2,STN_0001,STN_0002,0.693015
3,STN_0001,STN_0003,0.697196
4,STN_0005,STN_0008,0.727894
5,STN_0002,STN_0003,0.884400
6,STN_0001,STN_0005,0.937035
7,STN_0001,STN_0008,0.991646
8,STN_0002,STN_0004,1.027095
9,STN_0001,STN_0004,1.128362


In [22]:
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations

# --- output directory ---
out_dir = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# --- Preconditions checks ---
if "rides_df" not in globals():
    raise RuntimeError("rides_df not found in globals()")
if "top10" not in globals():
    raise RuntimeError("top10 not found in globals()")
if "canonical_mapping_df" not in globals():
    raise RuntimeError("canonical_mapping_df not found in globals()")

# canonical id column names expected in rides_df
start_col = "start_canonical_station_id"
end_col = "end_canonical_station_id"
if start_col not in rides_df.columns or end_col not in rides_df.columns:
    raise ValueError(f"rides_df must contain columns {start_col} and {end_col}")

# ensure top10 canonical ids list (still available if you need it later)
top10_cids = list(top10["canonical_station_id"].astype(str).tolist())
print("Top10 canonical ids (unchanged):", top10_cids)

# ----------------------------------------------------------------
# NEW: compute directed pair counts from the entire rides_df (no pre-filter)
# ----------------------------------------------------------------
directed_counts_all = (
    rides_df
    .groupby([start_col, end_col], dropna=False)
    .size()
    .rename("ride_count")
    .reset_index()
)

# Top 10 directed routes across the whole dataset
top10_directed_pairs = directed_counts_all.sort_values("ride_count", ascending=False).head(10).reset_index(drop=True)
print("Top 10 directed routes (from full dataset):")
display(top10_directed_pairs)

# If you still want a directed connectivity matrix restricted to the top10 stations (original behavior),
# you can build it from rides where both endpoints are in top10_cids, or from the top10 list of stations.
# But below I show how to build a directed matrix for the top10 stations (if desired):

# --- Optional: directed matrix for top10 stations (only if you want it) ---
mask_top10_endpoints = rides_df[start_col].astype(str).isin(top10_cids) & rides_df[end_col].astype(str).isin(top10_cids)
rides_top10_pairs = rides_df[mask_top10_endpoints].copy()
directed_counts_top10_matrix = (
    rides_top10_pairs
    .groupby([start_col, end_col], dropna=False)
    .size()
    .rename("ride_count")
    .reset_index()
)
directed_matrix = directed_counts_top10_matrix.pivot_table(index=start_col, columns=end_col, values="ride_count", fill_value=0)
directed_matrix = directed_matrix.reindex(index=top10_cids, columns=top10_cids, fill_value=0)

# ----------------------------------------------------------------
# Undirected aggregation computed from the full directed counts
# (treat A->B and B->A as the same pair)
# ----------------------------------------------------------------
def unordered_pair(a, b):
    a, b = str(a), str(b)
    return tuple(sorted([a, b]))

directed_counts_all["pair_key"] = directed_counts_all.apply(lambda r: unordered_pair(r[start_col], r[end_col]), axis=1)
undirected_agg = (
    directed_counts_all.groupby("pair_key")["ride_count"]
    .sum()
    .reset_index()
)
undirected_agg[["station_a", "station_b"]] = pd.DataFrame(undirected_agg["pair_key"].tolist(), index=undirected_agg.index)
undirected_agg = undirected_agg[["station_a", "station_b", "ride_count"]].sort_values("ride_count", ascending=False).reset_index(drop=True)

# Top 10 undirected pairs from the full dataset
top10_undirected_pairs = undirected_agg.head(10).copy()
print("Top 10 undirected pairs (from full dataset):")
display(top10_undirected_pairs)

# ----------------------------------------------------------------
# Pairwise distances for stations involved in the top10 directed routes
# (compute distances only for station ids that appear in the top10 routes)
# ----------------------------------------------------------------
# collect unique station ids from the top10 directed pairs
stations_in_top10_routes = pd.unique(top10_directed_pairs[[start_col, end_col]].astype(str).values.ravel())

# get canonical coords for those stations
canon_coords = canonical_mapping_df[["canonical_station_id", "canonical_lat", "canonical_lon"]].drop_duplicates("canonical_station_id")
canon_coords = canon_coords.set_index("canonical_station_id").reindex(stations_in_top10_routes)
if canon_coords["canonical_lat"].isna().any() or canon_coords["canonical_lon"].isna().any():
    print("Warning: some stations in top10 routes lack coordinates in canonical_mapping_df")

pairs = []
for a, b in combinations(stations_in_top10_routes, 2):
    lat_a, lon_a = canon_coords.loc[a, "canonical_lat"], canon_coords.loc[a, "canonical_lon"]
    lat_b, lon_b = canon_coords.loc[b, "canonical_lat"], canon_coords.loc[b, "canonical_lon"]
    if pd.isna(lat_a) or pd.isna(lon_a) or pd.isna(lat_b) or pd.isna(lon_b):
        dist_km = np.nan
    else:
        dist_km = haversine_km(lat_a, lon_a, lat_b, lon_b)
    pairs.append({"station_a": a, "station_b": b, "distance_km": float(dist_km) if not pd.isna(dist_km) else np.nan})
pairs_df = pd.DataFrame(pairs).sort_values("distance_km").reset_index(drop=True)

# ----------------------------------------------------------------
# Export results to CSV files (adjust filenames as you prefer)
# ----------------------------------------------------------------
directed_counts_all_csv = out_dir / "all_directed_pair_counts.csv"
top10_directed_csv = out_dir / "top10_directed_pairs_from_full_dataset.csv"
undirected_pairs_csv = out_dir / "all_undirected_pair_counts.csv"
top10_undirected_csv = out_dir / "top10_undirected_pairs_from_full_dataset.csv"
pairs_dist_csv = out_dir / "top10_routes_pairwise_distances_km.csv"
directed_matrix_csv = out_dir / "top10_directed_connectivity_matrix.csv"  # optional matrix file

directed_counts_all.to_csv(directed_counts_all_csv, index=False)
top10_directed_pairs.to_csv(top10_directed_csv, index=False)
undirected_agg.to_csv(undirected_pairs_csv, index=False)
top10_undirected_pairs.to_csv(top10_undirected_csv, index=False)
pairs_df.to_csv(pairs_dist_csv, index=False)
directed_matrix.reset_index().to_csv(directed_matrix_csv, index=False)

print("\nWrote CSVs to", out_dir)
print("Top 10 directed pairs file:", top10_directed_csv.name)
print("Top 10 undirected pairs file:", top10_undirected_csv.name)
print("Pairwise distances file:", pairs_dist_csv.name)

# display samples
print("\nTop 10 directed pairs (from full dataset):")
display(top10_directed_pairs)

print("\nTop 10 undirected pairs (from full dataset):")
display(top10_undirected_pairs)

print("\nPairwise distances among stations in top10 routes (sample):")
display(pairs_df.head(20))

Top10 canonical ids (unchanged): ['STN_0001', 'STN_0002', 'STN_0003', 'STN_0004', 'STN_0005', 'STN_0006', 'STN_0007', 'STN_0008', 'STN_0009', 'STN_0010']
Top 10 directed routes (from full dataset):


,start_canonical_station_id,end_canonical_station_id,ride_count
0,STN_0027,STN_0027,24493
1,STN_0070,STN_0279,9483
2,STN_0083,STN_0024,8462
3,STN_0025,STN_0013,8351
4,STN_0002,STN_0001,8121
5,STN_0070,STN_0163,8064
6,STN_0001,STN_0008,7993
7,STN_0426,STN_0426,7869
8,STN_0006,STN_0006,7730
9,STN_0008,STN_0001,7492


Top 10 undirected pairs (from full dataset):


,station_a,station_b,ride_count
0,STN_0027,STN_0027,24493
1,STN_0070,STN_0279,16417
2,STN_0001,STN_0008,15485
3,STN_0024,STN_0083,15128
4,STN_0070,STN_0163,14660
5,STN_0001,STN_0002,14024
6,STN_0013,STN_0025,14000
7,STN_0010,STN_0030,12777
8,STN_0027,STN_0499,12569
9,STN_0001,STN_0072,12092



Wrote CSVs to C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs
Top 10 directed pairs file: top10_directed_pairs_from_full_dataset.csv
Top 10 undirected pairs file: top10_undirected_pairs_from_full_dataset.csv
Pairwise distances file: top10_routes_pairwise_distances_km.csv

Top 10 directed pairs (from full dataset):


,start_canonical_station_id,end_canonical_station_id,ride_count
0,STN_0027,STN_0027,24493
1,STN_0070,STN_0279,9483
2,STN_0083,STN_0024,8462
3,STN_0025,STN_0013,8351
4,STN_0002,STN_0001,8121
5,STN_0070,STN_0163,8064
6,STN_0001,STN_0008,7993
7,STN_0426,STN_0426,7869
8,STN_0006,STN_0006,7730
9,STN_0008,STN_0001,7492



Top 10 undirected pairs (from full dataset):


,station_a,station_b,ride_count
0,STN_0027,STN_0027,24493
1,STN_0070,STN_0279,16417
2,STN_0001,STN_0008,15485
3,STN_0024,STN_0083,15128
4,STN_0070,STN_0163,14660
5,STN_0001,STN_0002,14024
6,STN_0013,STN_0025,14000
7,STN_0010,STN_0030,12777
8,STN_0027,STN_0499,12569
9,STN_0001,STN_0072,12092



Pairwise distances among stations in top10 routes (sample):


,station_a,station_b,distance_km
0,STN_0279,STN_0163,0.241648
1,STN_0083,STN_0024,0.550343
2,STN_0002,STN_0001,0.693015
3,STN_0070,STN_0083,0.820912
4,STN_0279,STN_0024,0.838432
5,STN_0070,STN_0279,0.891450
6,STN_0070,STN_0163,0.911741
7,STN_0279,STN_0083,0.926458
8,STN_0025,STN_0013,0.956809
9,STN_0001,STN_0008,0.991646


In [28]:
import sqlite3
import pandas as pd
from pathlib import Path

out_dir = Path(r"C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs")
out_dir.mkdir(parents=True, exist_ok=True)

s_col = "start_canonical_station_id"
e_col = "end_canonical_station_id"
if s_col not in rides_df.columns or e_col not in rides_df.columns:
    raise ValueError("rides_df must contain start_canonical_station_id and end_canonical_station_id")

# --- 1) start_count and end_count (memory efficient) ---
starts = rides_df[s_col].value_counts(dropna=True).rename("start_count")
ends   = rides_df[e_col].value_counts(dropna=True).rename("end_count")

traffic_df = pd.concat([starts, ends], axis=1).fillna(0)
traffic_df["start_count"] = traffic_df["start_count"].astype(int)
traffic_df["end_count"]   = traffic_df["end_count"].astype(int)
traffic_df["total_traffic"] = traffic_df["start_count"] + traffic_df["end_count"]
traffic_df = traffic_df.reset_index().rename(columns={"index": "canonical_station_id"})

# --- 2) Build distinct undirected pairs in a disk-backed SQLite table ---
# We'll create a small SQLite DB file in the outputs folder to store unique undirected pairs.
db_path = out_dir / "pairs_unique.db"
if db_path.exists():
    db_path.unlink()  # remove previous DB to start fresh

conn = sqlite3.connect(str(db_path))
cur = conn.cursor()

# table for unique undirected pairs (a < b lexicographically). Exclude self-loops for partner counting.
cur.execute("""
CREATE TABLE unique_pairs (
    a TEXT NOT NULL,
    b TEXT NOT NULL,
    UNIQUE(a,b)
)
""")
conn.commit()

# Insert normalized pairs in batches using INSERT OR IGNORE to keep only distinct pairs.
batch = []
batch_size = 100000  # tune this if needed
it = rides_df.itertuples(index=False, name=None)  # iterate rows without copying

count_rows = 0
for row in it:
    # row order depends on DataFrame columns; find start/end by name
    # safer to access by attribute names if available; fallback to index lookup
    try:
        start = getattr(row, s_col)
        end = getattr(row, e_col)
    except Exception:
        # fallback: assume tuple order matches DataFrame columns
        # find column indices
        start = row[rides_df.columns.get_loc(s_col)]
        end = row[rides_df.columns.get_loc(e_col)]

    if start is None or end is None:
        continue
    a = str(start).strip()
    b = str(end).strip()
    if a == "" or b == "":
        continue
    if a == b:
        # skip self-loop for partner counting (self is not a distinct partner)
        continue
    # normalize order so (a,b) with a <= b
    if a <= b:
        pair = (a, b)
    else:
        pair = (b, a)
    batch.append(pair)
    count_rows += 1

    if len(batch) >= batch_size:
        cur.executemany("INSERT OR IGNORE INTO unique_pairs(a,b) VALUES (?,?)", batch)
        conn.commit()
        batch = []

# insert remaining
if batch:
    cur.executemany("INSERT OR IGNORE INTO unique_pairs(a,b) VALUES (?,?)", batch)
    conn.commit()
    batch = []

# --- 3) Compute unique partner counts from unique_pairs table ---
# For each pair (a,b) both a and b gain one partner (the other)
query = """
SELECT station, COUNT(*) AS unique_partners FROM (
    SELECT a AS station, b AS partner FROM unique_pairs
    UNION ALL
    SELECT b AS station, a AS partner FROM unique_pairs
) GROUP BY station
"""
partner_counts_df = pd.read_sql_query(query, conn)
conn.close()

# --- 4) Merge partner counts into traffic_df and finalize ---
result_df = traffic_df.merge(partner_counts_df.rename(columns={"station":"canonical_station_id"}), 
                             on="canonical_station_id", how="left").fillna({"unique_partners": 0})
result_df["unique_partners"] = result_df["unique_partners"].astype(int)

# optional: attach canonical coords if available
if {"canonical_station_id","canonical_lat","canonical_lon"}.issubset(set(canonical_mapping_df.columns)):
    coords = canonical_mapping_df[["canonical_station_id","canonical_lat","canonical_lon"]].drop_duplicates("canonical_station_id")
    result_df = result_df.merge(coords, on="canonical_station_id", how="left")

# --- 5) Save and show top 10 ---
out_csv = out_dir / "canonical_stations_total_traffic_all.csv"
result_df.to_csv(out_csv, index=False)

top10 = result_df.sort_values("total_traffic", ascending=False).head(10).reset_index(drop=True)

print("Saved full station traffic table to:", out_csv)
print("\nTop 10 canonical stations by total_traffic")
display(top10)

Saved full station traffic table to: C:\Users\Daniel Loh\Documents\NUS-ISS_EBAC\BDA\BDA Project Module\outputs\canonical_stations_total_traffic_all.csv

Top 10 canonical stations by total_traffic


,canonical_station_id,start_count,end_count,total_traffic,unique_partners,canonical_lat,canonical_lon
0,STN_0001,249555,224547,474102,1124,45.524353,-73.581432
1,STN_0002,229304,201571,430875,1128,45.519410,-73.586850
2,STN_0003,180266,146026,326292,1130,45.527154,-73.589439
3,STN_0004,152579,141305,293884,1086,45.515228,-73.575096
4,STN_0005,137603,136216,273819,1081,45.532449,-73.584770
5,STN_0006,106102,158995,265097,1143,45.507610,-73.551836
6,STN_0007,127581,133950,261531,1109,45.500380,-73.575070
7,STN_0008,126015,121818,247833,1030,45.532218,-73.575431
8,STN_0009,122592,119676,242268,1107,45.496840,-73.579241
9,STN_0010,107896,133600,241496,1105,45.523845,-73.552366
